# Ingestão de Dados Hidrológicos Projeto Iguaçu
## Hidrovia Paraguai-Paraná (Estações: Asuncíon e Ladário)

Notebook responsável pela extração dos dados de níveis fluviais da Estação de Ladário da Hidrovia do Rio Paraguai. 

Os dados são obtidos via API da Nasa Power.

Destino:
- Parquet - Volume:
  - Asuncíon: `01_bronze/nasa_power/asuncion`
  - Ladário: `01_bronze/nasa_power/ladario`
- Tabela:
  - Asuncíon: `01_bronze.nasa_power_asuncion`
  - Ladário: `01_bronze.nasa_power_ladario`

Dados climaticos das estações
- Asuncion: 
  - Latitude: -25.2675
  - Longitude: -57.6408
- Ladário: 
  - Latitude: -19.0017
  - Longitude: -57.5942

Variáveis climaticas: https://power.larc.nasa.gov/
- 🌡️ Temperatura:
    - T2M: Temperatura média do ar a 2 metros (°C)
    - T2M_MAX: Temperatura máxima diária a 2 metros (°C)
    - T2M_MIN: Temperatura mínima diária a 2 metros (°C)
    - TS: Temperatura da superfície terrestre (°C)
- 🌧️ Precipitação e Umidade
    - PRECTOT: Precipitação total diária (mm)
    - RH2M: Umidade relativa a 2 metros (%)

Schema:

```python
root
 |-- DATE: date (nullable = true)
 |-- T2M: double (nullable = true)
 |-- TS: double (nullable = true)
 |-- RH2M: double (nullable = true)
 |-- T2M_MIN: double (nullable = true)
 |-- T2M_MAX: double (nullable = true)
 |-- PRECTOTCORR: double (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- INGESTION_DATE: date (nullable = false)
```

In [0]:
## libs python

import logging
import requests
import pandas as pd

In [0]:
## libs pyspark

from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

In [0]:
# Constantes
nasa_api_base_url = 'https://power.larc.nasa.gov'
api_url = f'{nasa_api_base_url}/api/temporal/daily/point'
community = 'sb'  # Agricultura (ag) ou Energia (sb)
climatology_variables = ['T2M', 'T2M_MAX', 'T2M_MIN', 'TS', 'PRECTOT', 'RH2M']
years = [2025]

asuncion_latitude = -25.2675
asuncion_longitude = -57.6408
ladario_latitude = -19.0017
ladario_longitude = -57.5942

asuncion_parquet_path = '/Volumes/iguacu_lakehouse/01_bronze/nasa_power/asuncion'
ladario_parquet_path = '/Volumes/iguacu_lakehouse/01_bronze/nasa_power/ladario'
asuncion_table_name = 'iguacu_lakehouse.01_bronze.nasa_power_asuncion'
ladario_table_name = 'iguacu_lakehouse.01_bronze.nasa_power_ladario'

In [0]:
def setup_logging(name: str = "bronze_ingestion") -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
        logger.addHandler(handler)
    
    return logger

logger = setup_logging()

In [0]:
def get_climatology_data(
  api_url: str, 
  latitude: float, 
  longitiude: float,
  year: int, 
  community: str, 
  climatology_variables: list, 
) -> DataFrame:
  
  url = (
    f'{api_url}?start={year}0101&end={year}1231'
    f'&latitude={latitude}&longitude={longitiude}'
    f'&community={community}'
    f'&parameters={",".join(climatology_variables)}'
    '&format=json'
  )
  
  response = requests.get(url)
  if response.status_code == 200:
    data = response.json()
    records = data['properties']['parameter']
    
    # Montar DataFrame
    data_df = pd.DataFrame(records)
    data_df.reset_index(inplace=True)
    data_df['index'] = pd.to_datetime(data_df['index'], format='%Y%m%d').dt.date
    data_df.rename({'index':'DATE'}, axis=1, inplace=True)
    
    spark_df = spark.createDataFrame(data_df).withColumn(
        'MONTH',
        F.month(F.col('DATE'))
    ).withColumn(
        'YEAR',
        F.year(F.col('DATE'))
    ).withColumn(
        'INGESTION_DATE',
        F.current_date()
    )

    return spark_df   

  else:
    logger.error("Erro na requisição:", response.status_code)

  return None


In [0]:
def merge_to_delta(df: DataFrame, table_name: str) -> None:
    # merger dos dados
    delta_table = DeltaTable.forName(spark, table_name)
    (
        delta_table.alias('target')
        .merge(
            df.alias('source'),
            'target.DATE = source.DATE'
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    logger.info(f'MERGE concluído em {table_name} com %s {df.count()}')
     



extract Asuncion NASA POWER Data


In [0]:
# Buscando os dados de Asuncíon
for year in years:
    asuncion_df = get_climatology_data(
        api_url, 
        asuncion_latitude, 
        asuncion_longitude, 
        year, 
        community, 
        climatology_variables
    )

    #salvar arquivo parquet
    (
        asuncion_df
        .write
        .partitionBy(['year', 'month'])
        .mode('append')
        .format('parquet')
        .save(asuncion_parquet_path)
    )
    
    #salvar tabela delta
    merge_to_delta(asuncion_df, asuncion_table_name)

extract Ladario NASA POWER Data

In [0]:
# Buscando os dados de Ladario
for year in years:
    ladario_df = get_climatology_data(
        api_url, 
        ladario_latitude, 
        ladario_longitude, 
        year, 
        community, 
        climatology_variables
    )

    #salvar arquivo parquet
    (
        ladario_df
        .write
        .partitionBy(['year', 'month'])
        .mode('append')
        .format('parquet')
        .save(ladario_parquet_path)
    )
    
    #salvar tabela delta
    merge_to_delta(ladario_df, ladario_table_name)